Load The Dataset

In [0]:
%python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeltaLakeAssignment") \
    .getOrCreate()

master = spark.read.option("header",True).csv("/Workspace/Users/dhruvaa866@gmail.com/customer_master.csv")

incremental = spark.read.option("header",True).csv("/Workspace/Users/dhruvaa866@gmail.com/customer_incremental.csv")

master.show()
incremental.show()

+-----------+-------------+------+---+-------------+-----+--------------------+----------+----------+--------+
|customer_id|         name|gender|age|         city|state|               email|     phone| join_date|  status|
+-----------+-------------+------+---+-------------+-----+--------------------+----------+----------+--------+
|       1001|Alice Johnson|Female| 28|     New York|   NY|alice.johnson@gma...|9876543210|2022-01-15|  Active|
|       1002|    Bob Smith|  Male| 35|      Chicago|   IL| bob.smith@gmail.com|9876543211|2022-03-10|  Active|
|       1003|Charlie Brown|  Male| 40|       Dallas|   TX|charlie.brown@gma...|9876543212|2021-12-05|Inactive|
|       1004| David Wilson|  Male| 30|       Boston|   MA|david.wilson@gmai...|9876543213|2023-02-18|  Active|
|       1005|   Emma Davis|Female| 26|        Miami|   FL|emma.davis@gmail.com|9876543214|2022-05-12|  Active|
|       1006| Frank Miller|  Male| 45|      Seattle|   WA|frank.miller@gmai...|9876543215|2021-08-20|Inactive|
|

Clean the Data

In [0]:
%python
master = master.dropDuplicates().na.drop()

incremental = incremental.dropDuplicates().na.drop()

Create Delta Table

In [0]:
%python
master.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.customer_master_delta")

In [0]:
%python
spark.sql("""
SELECT * 
FROM workspace.default.customer_master_delta
""").show()

+-----------+-------------+------+---+-------------+-----+--------------------+----------+----------+--------+
|customer_id|         name|gender|age|         city|state|               email|     phone| join_date|  status|
+-----------+-------------+------+---+-------------+-----+--------------------+----------+----------+--------+
|       1006| Frank Miller|  Male| 45|      Seattle|   WA|frank.miller@gmai...|9876543215|2021-08-20|Inactive|
|       1010|   Jack White|  Male| 41|      Phoenix|   AZ|jack.white@gmail.com|9876543219|2022-06-30|Inactive|
|       1008|  Henry Clark|  Male| 29|       Denver|   CO|henry.clark@gmail...|9876543217|2022-10-11|  Active|
|       1001|Alice Johnson|Female| 28|     New York|   NY|alice.johnson@gma...|9876543210|2022-01-15|  Active|
|       1009|   Ivy Thomas|Female| 38|      Atlanta|   GA|ivy.thomas@gmail.com|9876543218|2021-11-25|  Active|
|       1004| David Wilson|  Male| 30|       Boston|   MA|david.wilson@gmai...|9876543213|2023-02-18|  Active|
|

Load the Delta Table

In [0]:
%python
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(
    spark,
    "workspace.default.customer_master_delta"
)

Perform the Merge

In [0]:
%python
(
    deltaTable.alias("old")
    .merge(
        incremental.alias("new"),
        "old.customer_id = new.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Final Result

In [0]:
%python
result = spark.table(
    "workspace.default.customer_master_delta"
)

display(result)

print("Final Row Count:", result.count())

customer_id,name,gender,age,city,state,email,phone,join_date,status
1006,Frank Miller,Male,45,Seattle,WA,frank.miller@gmail.com,9876543215,2021-08-20,Inactive
1010,Jack White,Male,41,Phoenix,AZ,jack.white@gmail.com,9876543219,2022-06-30,Inactive
1008,Henry Clark,Male,29,Denver,CO,henry.clark@gmail.com,9876543217,2022-10-11,Active
1001,Alice Johnson,Female,28,New York,NY,alice.johnson@gmail.com,9876543210,2022-01-15,Active
1009,Ivy Thomas,Female,38,Atlanta,GA,ivy.thomas@gmail.com,9876543218,2021-11-25,Active
1003,Charlie Brown,Male,40,Dallas,TX,charlie.brown@gmail.com,9876543212,2021-12-05,Inactive
1005,Emma Davis,Female,26,Miami,FL,emma.davis@gmail.com,9876543214,2022-05-12,Active
1013,Mia Walker,Female,25,Las Vegas,NV,mia.walker@gmail.com,9876543222,2024-03-01,Active
1002,Bob Smith,Male,35,Seattle,WA,bob.smith@gmail.com,9876543211,2022-03-10,Active
1012,Liam Harris,Male,33,Portland,OR,liam.harris@gmail.com,9876543221,2024-02-20,Active


Final Row Count: 13
